# 02. 学習用データセットの作成

このnotebookでは、YouTube Data APIから取得した raw CSV を読み込み、機械学習に使うための processed CSV を作成する。

主な処理は以下。

- raw CSVの読み込み
- 学習対象データと other_candidates の分離
- title / description の欠損処理
- 分類用テキスト列の作成
- 学習用CSVとして保存
- ジャンル件数・文字数・チャンネル偏りの確認

## 1. ライブラリ読み込みとパス設定

raw CSV の保存場所と、processed CSV の出力先を定義する。

In [1]:
import pandas as pd
from pathlib import Path

cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_PATH = PROJECT_ROOT / "data/raw/videos_raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_PATH = PROCESSED_DIR / "videos_labeled.csv"
OTHER_PATH = PROCESSED_DIR / "other_candidates.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("exists:", RAW_PATH.exists())

PROJECT_ROOT: /Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc
RAW_PATH: /Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/raw/videos_raw.csv
exists: True


## 2. raw CSVの読み込み

前工程で作成した `data/raw/videos_raw.csv` を読み込む。  
このCSVは、1行を1本のYouTube動画として扱っている。

In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(df_raw.shape)
df_raw.head()

(464, 21)


,source_playlist_id,source_playlist_name,genre,use_for_training,video_id,url,is_available,title,description,channel_id,...,tags,category_id,duration,published_at,view_count,like_count,comment_count,default_language,default_audio_language,fetched_at
0,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,ktzbPzmAnnw,https://www.youtube.com/watch?v=ktzbPzmAnnw,True,「胎児の夢」 feat.初音ミク,胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見ているっていう説があるらしいです。\...,UCVJns279mOOd92lWVQl_HlA,...,"[""#VOCALOID"", ""#初音ミク""]",10,PT3M33S,2020-03-31T16:35:22Z,425482,18481.0,176.0,ja,NaN,2026-06-20T13:05:06.655004+00:00
1,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,arlXG7TVdPk,https://www.youtube.com/watch?v=arlXG7TVdPk,True,天使じゃないよ/初音ミク,他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクですよね\n\n歌：初音ミク\...,UCF9BrkN5Z5tqAwuj_s0519g,...,[],22,PT3M36S,2024-06-21T10:00:05Z,140917,7542.0,134.0,ja,NaN,2026-06-20T13:05:06.655004+00:00
2,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,DNltbDXaKw8,https://www.youtube.com/watch?v=DNltbDXaKw8,True,きおくをみたの/初音ミク(まかろり×いのうつはSA),生きるって、何だろうね。\n\n作詞作曲：まかろり\n(https://twitter.co...,UCB7DcMLTrdCeCENzfZIfKLQ,...,[],22,PT3M45S,2024-09-29T11:00:22Z,133008,5441.0,176.0,ja,ja,2026-06-20T13:05:06.655004+00:00
3,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,JqvE2fxm80U,https://www.youtube.com/watch?v=JqvE2fxm80U,True,夢憂鬱 / 初音ミク,VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n\n₁₀₄は₆番線さん...,UCwqj1NywsBPbd5QZvHf-yEg,...,[],22,PT3M28S,2024-07-08T10:00:06Z,165424,9875.0,145.0,ja,en,2026-06-20T13:05:06.655004+00:00
4,PLpssoRoNCKKzuvZMpCmHxh7i6Qu5xYoLY,YTClassify_music,music,True,Pmj5CGPaUQ8,https://www.youtube.com/watch?v=Pmj5CGPaUQ8,True,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",▶Music れーしあ\nhttps://x.com/Re_siadesu\n\n▶Illu...,UC90a5GK8XetNIlNF0kPj7yA,...,"[""八月"", ""僕らの犯した間違いの答え合わせ"", ""ボカロ"", ""夏 ボカロ"", ""カゼヒ...",10,PT3M44S,2024-08-23T11:00:27Z,93070,6715.0,220.0,ja,NaN,2026-06-20T13:05:06.655004+00:00


## 3. rawデータの基本確認

processed CSVを作成する前に、以下を確認する。

- 行数・列数
- ジャンル別件数
- 学習対象フラグ
- 動画取得可否
- video_id の重複

ここで想定外の欠損や重複がないかを確認する。

In [3]:
print("columns:")
print(df_raw.columns.tolist())

print("\nshape:")
print(df_raw.shape)

print("\ngenre counts:")
print(df_raw["genre"].value_counts())

print("\nuse_for_training:")
print(df_raw["use_for_training"].value_counts(dropna=False))

print("\nis_available:")
print(df_raw["is_available"].value_counts(dropna=False))

print("\nduplicated video_id:")
print(df_raw["video_id"].duplicated().sum())

columns:
['source_playlist_id', 'source_playlist_name', 'genre', 'use_for_training', 'video_id', 'url', 'is_available', 'title', 'description', 'channel_id', 'channel_title', 'tags', 'category_id', 'duration', 'published_at', 'view_count', 'like_count', 'comment_count', 'default_language', 'default_audio_language', 'fetched_at']

shape:
(464, 21)

genre counts:
genre
music               123
study               111
game                107
cooking             102
other_candidates     21
Name: count, dtype: int64

use_for_training:
use_for_training
True     443
False     21
Name: count, dtype: int64

is_available:
is_available
True    464
Name: count, dtype: int64

duplicated video_id:
0


## 4. 学習用データと観察用データの分離

`use_for_training == True` の動画を学習用データとして使用する。  
`other_candidates` は、今回は学習には使わず、学習済みモデルに入力したときにどのジャンルへ寄るかを観察するために分けて保存する。

In [4]:
def to_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s
    
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        })
    )

In [5]:
df = df_raw.copy()

df["is_available_bool"] = to_bool_series(df["is_available"])
df["use_for_training_bool"] = to_bool_series(df["use_for_training"])

df_train = df[
    (df["is_available_bool"] == True) &
    (df["use_for_training_bool"] == True)
].copy()

df_other = df[
    (df["is_available_bool"] == True) &
    (df["use_for_training_bool"] == False)
].copy()

print("train:", df_train.shape)
print("other:", df_other.shape)

print("\ntrain genre counts:")
print(df_train["genre"].value_counts())

print("\nother genre counts:")
print(df_other["genre"].value_counts())

train: (443, 23)
other: (21, 23)

train genre counts:
genre
music      123
study      111
game       107
cooking    102
Name: count, dtype: int64

other genre counts:
genre
other_candidates    21
Name: count, dtype: int64


## 5. テキスト列の欠損処理

分類モデルでは `title` と `description` を結合したテキストを使用する。  
欠損値があると文字列結合やTF-IDF変換で扱いづらいため、空文字に変換する。


In [6]:
TEXT_COLUMNS = ["title", "description", "channel_title", "tags"]

for col in TEXT_COLUMNS:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna("").astype(str)
    if col in df_other.columns:
        df_other[col] = df_other[col].fillna("").astype(str)

## 6. 分類用テキスト列の作成

まずはベースラインとして、以下のようにタイトルと概要欄を単純に結合する。

```text
text_base = title + " " + description
```
また、YouTube動画ではタイトルの方がジャンル情報を強く含む可能性があるため、タイトルを複数回連結した列も作成する。

In [7]:
df_train["text_base"] = (
    df_train["title"].str.strip() + " " + df_train["description"].str.strip()
).str.strip()

df_other["text_base"] = (
    df_other["title"].str.strip() + " " + df_other["description"].str.strip()
).str.strip()

In [8]:
def make_weighted_text(row, title_weight: int) -> str:
    title = str(row["title"]).strip()
    description = str(row["description"]).strip()
    return " ".join([title] * title_weight + [description]).strip()

for w in [1, 2, 3]:
    df_train[f"text_title_weight_{w}"] = df_train.apply(
        make_weighted_text,
        axis=1,
        title_weight=w
    )
    df_other[f"text_title_weight_{w}"] = df_other.apply(
        make_weighted_text,
        axis=1,
        title_weight=w
    )

## 7. 文字数と空テキストの確認

作成したテキスト列について、空文字になっている動画がないか確認する。  
また、タイトル・概要欄・結合テキストの文字数分布を確認し、極端に短い/長いデータがないかを見る。

In [9]:
df_train["title_length"] = df_train["title"].str.len()
df_train["description_length"] = df_train["description"].str.len()
df_train["text_length"] = df_train["text_base"].str.len()

df_other["title_length"] = df_other["title"].str.len()
df_other["description_length"] = df_other["description"].str.len()
df_other["text_length"] = df_other["text_base"].str.len()

print("empty text count:")
print((df_train["text_base"].str.len() == 0).sum())

print("\ntext length describe:")
display(df_train[["title_length", "description_length", "text_length"]].describe())

empty text count:
0

text length describe:


,title_length,description_length,text_length
count,443.000000,443.000000,443.000000
mean,36.442438,748.988713,786.367946
std,19.685381,775.372176,781.608333
min,1.000000,0.000000,12.000000
25%,22.000000,192.500000,223.500000
50%,32.000000,505.000000,544.000000
75%,47.000000,1033.500000,1078.000000
max,98.000000,4909.000000,4966.000000


In [10]:
df_train = df_train[df_train["text_base"].str.len() > 0].copy()
df_other = df_other[df_other["text_base"].str.len() > 0].copy()

## 8. processed CSVとして保存する列の選定

学習に直接使う列だけでなく、後で誤分類分析を行うために以下の情報も残す。

- video_id
- url
- title
- description
- channel_title
- tags
- duration
- view_count
- text_base

特に `url` と `title` は、誤分類した動画を後から確認するために重要である。

In [11]:
KEEP_COLUMNS = [
    "video_id",
    "url",
    "genre",
    "title",
    "description",
    "channel_id",
    "channel_title",
    "tags",
    "category_id",
    "duration",
    "published_at",
    "view_count",
    "like_count",
    "comment_count",
    "default_language",
    "default_audio_language",
    "fetched_at",
    "text_base",
    "text_title_weight_1",
    "text_title_weight_2",
    "text_title_weight_3",
    "title_length",
    "description_length",
    "text_length",
]

existing_columns = [col for col in KEEP_COLUMNS if col in df_train.columns]

df_train_processed = df_train[existing_columns].copy()
df_other_processed = df_other[existing_columns].copy()

print(df_train_processed.shape)
df_train_processed.head()

(443, 24)


,video_id,url,genre,title,description,channel_id,channel_title,tags,category_id,duration,...,default_language,default_audio_language,fetched_at,text_base,text_title_weight_1,text_title_weight_2,text_title_weight_3,title_length,description_length,text_length
0,ktzbPzmAnnw,https://www.youtube.com/watch?v=ktzbPzmAnnw,music,「胎児の夢」 feat.初音ミク,胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見ているっていう説があるらしいです。\...,UCVJns279mOOd92lWVQl_HlA,水都,"[""#VOCALOID"", ""#初音ミク""]",10,PT3M33S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見て...,「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見て...,「胎児の夢」 feat.初音ミク 「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間...,「胎児の夢」 feat.初音ミク 「胎児の夢」 feat.初音ミク 「胎児の夢」 feat....,16,213,230
1,arlXG7TVdPk,https://www.youtube.com/watch?v=arlXG7TVdPk,music,天使じゃないよ/初音ミク,他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクですよね\n\n歌：初音ミク\...,UCF9BrkN5Z5tqAwuj_s0519g,越前,[],22,PT3M36S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクです...,天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクです...,天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられ...,天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 他人に理想を押...,12,171,184
2,DNltbDXaKw8,https://www.youtube.com/watch?v=DNltbDXaKw8,music,きおくをみたの/初音ミク(まかろり×いのうつはSA),生きるって、何だろうね。\n\n作詞作曲：まかろり\n(https://twitter.co...,UCB7DcMLTrdCeCENzfZIfKLQ,まかろり,[],22,PT3M45S,...,ja,ja,2026-06-20T13:05:06.655004+00:00,きおくをみたの/初音ミク(まかろり×いのうつはSA) 生きるって、何だろうね。\n\n作詞作...,きおくをみたの/初音ミク(まかろり×いのうつはSA) 生きるって、何だろうね。\n\n作詞作...,きおくをみたの/初音ミク(まかろり×いのうつはSA) きおくをみたの/初音ミク(まかろり×い...,きおくをみたの/初音ミク(まかろり×いのうつはSA) きおくをみたの/初音ミク(まかろり×い...,26,197,224
3,JqvE2fxm80U,https://www.youtube.com/watch?v=JqvE2fxm80U,music,夢憂鬱 / 初音ミク,VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n\n₁₀₄は₆番線さん...,UCwqj1NywsBPbd5QZvHf-yEg,ukaihi,[],22,PT3M28S,...,ja,en,2026-06-20T13:05:06.655004+00:00,夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n...,夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n...,夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌...,夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク VocaDuo2024にて...,10,309,320
4,Pmj5CGPaUQ8,https://www.youtube.com/watch?v=Pmj5CGPaUQ8,music,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",▶Music れーしあ\nhttps://x.com/Re_siadesu\n\n▶Illu...,UC90a5GK8XetNIlNF0kPj7yA,れーしあ / Re_shia,"[""八月"", ""僕らの犯した間違いの答え合わせ"", ""ボカロ"", ""夏 ボカロ"", ""カゼヒ...",10,PT3M44S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",70,853,924


## 9. processed CSVの保存

学習用データを `data/processed/videos_labeled.csv` に保存する。  
また、学習には使わない `other_candidates` は別ファイルとして保存する。

In [12]:
df_train_processed.to_csv(PROCESSED_PATH, index=False)
df_other_processed.to_csv(OTHER_PATH, index=False)

print(f"saved train: {PROCESSED_PATH}")
print(f"saved other: {OTHER_PATH}")

saved train: /Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/processed/videos_labeled.csv
saved other: /Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/processed/other_candidates.csv


## 10. 保存後の確認

保存したCSVを再度読み込み、以下を確認する。

- 正しく読み込めるか
- ジャンル別件数が想定通りか
- video_id の重複がないか

ここまで確認できれば、次のEDA・分類実験に進める。

In [13]:
df_check = pd.read_csv(PROCESSED_PATH)

print(df_check.shape)
print(df_check["genre"].value_counts())
print(df_check["video_id"].duplicated().sum())

df_check.head()

(443, 24)
genre
music      123
study      111
game       107
cooking    102
Name: count, dtype: int64
0


,video_id,url,genre,title,description,channel_id,channel_title,tags,category_id,duration,...,default_language,default_audio_language,fetched_at,text_base,text_title_weight_1,text_title_weight_2,text_title_weight_3,title_length,description_length,text_length
0,ktzbPzmAnnw,https://www.youtube.com/watch?v=ktzbPzmAnnw,music,「胎児の夢」 feat.初音ミク,胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見ているっていう説があるらしいです。\...,UCVJns279mOOd92lWVQl_HlA,水都,"[""#VOCALOID"", ""#初音ミク""]",10,PT3M33S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見て...,「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間、生物の進化という壮絶な悪夢を見て...,「胎児の夢」 feat.初音ミク 「胎児の夢」 feat.初音ミク 胎児は母親の胎内にいる間...,「胎児の夢」 feat.初音ミク 「胎児の夢」 feat.初音ミク 「胎児の夢」 feat....,16,213,230
1,arlXG7TVdPk,https://www.youtube.com/watch?v=arlXG7TVdPk,music,天使じゃないよ/初音ミク,他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクですよね\n\n歌：初音ミク\...,UCF9BrkN5Z5tqAwuj_s0519g,越前,[],22,PT3M36S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクです...,天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられるのも等しくグロテスクです...,天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 他人に理想を押し付けるのも、押し付けられ...,天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 天使じゃないよ/初音ミク 他人に理想を押...,12,171,184
2,DNltbDXaKw8,https://www.youtube.com/watch?v=DNltbDXaKw8,music,きおくをみたの/初音ミク(まかろり×いのうつはSA),生きるって、何だろうね。\n\n作詞作曲：まかろり\n(https://twitter.co...,UCB7DcMLTrdCeCENzfZIfKLQ,まかろり,[],22,PT3M45S,...,ja,ja,2026-06-20T13:05:06.655004+00:00,きおくをみたの/初音ミク(まかろり×いのうつはSA) 生きるって、何だろうね。\n\n作詞作...,きおくをみたの/初音ミク(まかろり×いのうつはSA) 生きるって、何だろうね。\n\n作詞作...,きおくをみたの/初音ミク(まかろり×いのうつはSA) きおくをみたの/初音ミク(まかろり×い...,きおくをみたの/初音ミク(まかろり×いのうつはSA) きおくをみたの/初音ミク(まかろり×い...,26,197,224
3,JqvE2fxm80U,https://www.youtube.com/watch?v=JqvE2fxm80U,music,夢憂鬱 / 初音ミク,VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n\n₁₀₄は₆番線さん...,UCwqj1NywsBPbd5QZvHf-yEg,ukaihi,[],22,PT3M28S,...,ja,en,2026-06-20T13:05:06.655004+00:00,夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n...,夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌唱verです。\n\n...,夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク VocaDuo2024にて制作した曲の初音ミク歌...,夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク 夢憂鬱 / 初音ミク VocaDuo2024にて...,10,309,320
4,Pmj5CGPaUQ8,https://www.youtube.com/watch?v=Pmj5CGPaUQ8,music,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",▶Music れーしあ\nhttps://x.com/Re_siadesu\n\n▶Illu...,UC90a5GK8XetNIlNF0kPj7yA,れーしあ / Re_shia,"[""八月"", ""僕らの犯した間違いの答え合わせ"", ""ボカロ"", ""夏 ボカロ"", ""カゼヒ...",10,PT3M44S,...,ja,NaN,2026-06-20T13:05:06.655004+00:00,"八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...","八月、僕らの犯した間違いの答え合わせ/ カゼヒキ - August, the answer...",70,853,924


## 11. チャンネル偏りの確認

同じチャンネルの動画が多い場合、モデルがジャンルそのものではなく、チャンネル名や概要欄テンプレートを学習してしまう可能性がある。

今回はPoCなので完全には除外しないが、データの偏りとして把握し、READMEの考察に反映する。

In [14]:
channel_counts = (
    df_train_processed
    .groupby(["genre", "channel_title"])
    .size()
    .reset_index(name="count")
    .sort_values(["genre", "count"], ascending=[True, False])
)

channel_counts.head(30)

,genre,channel_title,count
32,cooking,だれウマ【料理研究家】,4
49,cooking,テイストメイド ジャパン,4
5,cooking,George ジョージ吉田,2
11,cooking,kattyanneru/かっちゃんねる,2
16,cooking,おすぎ(管理栄養士),2
21,cooking,きまま.Kimama ch★.,2
22,cooking,くまの限界食堂,2
23,cooking,けんた食堂,2
30,cooking,せこまる食堂,2
39,cooking,ぴーきちごはん,2


In [15]:
for genre, group in channel_counts.groupby("genre"):
    print(f"\n=== {genre} ===")
    display(group.head(10))


=== cooking ===


,genre,channel_title,count
32,cooking,だれウマ【料理研究家】,4
49,cooking,テイストメイド ジャパン,4
5,cooking,George ジョージ吉田,2
11,cooking,kattyanneru/かっちゃんねる,2
16,cooking,おすぎ(管理栄養士),2
21,cooking,きまま.Kimama ch★.,2
22,cooking,くまの限界食堂,2
23,cooking,けんた食堂,2
30,cooking,せこまる食堂,2
39,cooking,ぴーきちごはん,2



=== game ===


,genre,channel_title,count
88,game,HikakinGames,2
110,game,zubu,2
114,game,さなちゃんねる,2
120,game,ちゃあ/chaa's,2
129,game,のばまんゲームス,2
132,game,ひぬ【マイクラダンジョンズ&ダンジョンズ2攻略解説ch】,2
148,game,カナメとハルキー,2
165,game,主役は我々だ!,2
174,game,最後の壁【バグ動画メイン】,2
83,game,ニート部,1



=== music ===


,genre,channel_title,count
210,music,NY channel,2
214,music,PAS TASTA,2
217,music,SPACE SHOWER,2
223,music,TWICE - Topic,2
241,music,mukadetokyo,2
259,music,いのうつはSA,2
260,music,お柴鉱脈 Oshibacomyaku,2
266,music,まかろり,2
292,music,電ǂ鯨,2
181,music,- LΛMPLIGHT,1



=== study ===


,genre,channel_title,count
295,study,3Blue1BrownJapan,3
296,study,AKITOの勉強チャンネル,3
335,study,「ただよび」文系チャンネル,3
336,study,【TOEIC対策】猛牛ちゃんねる,3
362,study,予備校のノリで学ぶ「大学の数学・物理」,3
376,study,渡邉究 数学科教員,3
298,study,Amazon Web Services Japan 公式,2
306,study,Google Cloud Japan,2
310,study,Kazu Sekizawa - Nuclear Physicist at Science T...,2
311,study,Keiichi Yamasaki,2


In [16]:
print("processed dataset created")

print("\ntrain shape:")
print(df_train_processed.shape)

print("\ngenre counts:")
print(df_train_processed["genre"].value_counts())

print("\nother shape:")
print(df_other_processed.shape)

print("\nfiles:")
print(PROCESSED_PATH)
print(OTHER_PATH)

processed dataset created

train shape:
(443, 24)

genre counts:
genre
music      123
study      111
game       107
cooking    102
Name: count, dtype: int64

other shape:
(21, 24)

files:
/Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/processed/videos_labeled.csv
/Users/nishiyamaharuya/Documents/coding/youtube_analyzer/youtube-playlist-classifier-poc/data/processed/other_candidates.csv


## まとめ

raw CSV から学習用の processed CSV を作成した。

確認結果は以下。

- 学習対象データは443件
- `text_base` が空の動画は0件
- `title` と `description` を結合した分類用テキストを作成
- タイトル重み付け実験用に `text_title_weight_1` / `2` / `3` を作成
- `other_candidates` は学習には使わず、観察用データとして別保存
- チャンネル偏りを確認したが、特定チャンネルへの極端な集中は見られなかった

次のnotebookでは、processed CSVを使ってEDAを行う。
ジャンル別件数、文字数分布、頻出語、概要欄ノイズなどを確認する。